<a href="https://colab.research.google.com/github/BorysZhyhalo/goit-rdb-fp/blob/main/goit_rdb_fp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Relational Databases — Final Project

## Task 1. Database Setup and Data Import

In [54]:
!pip install -q pgserver==0.1.4 \
    --python-version 3.12 \
    --only-binary=:all: \
    --target=/usr/local/lib/python3.13/dist-packages

In [55]:
%pip install -q "psycopg[binary]" sqlalchemy pandas pyarrow requests

In [56]:
import os
import sys
import pandas as pd
import pgserver

from urllib.request import urlretrieve
from sqlalchemy import create_engine, text
from google.colab import files

print("Python:", sys.version)

pg = pgserver.get_server('/tmp/goit_rdb_fp', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [57]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE SCHEMA IF NOT EXISTS pandemic;
        SET search_path TO pandemic;
    """))

    result = conn.execute(text("SHOW search_path;")).scalar_one()
    print("Current schema search path:", result)

Current schema search path: pandemic


In [58]:
CSV_URL = (
    "https://raw.githubusercontent.com/"
    "BorysZhyhalo/goit-rdb-fp/main/infectious_cases.csv"
)

df = pd.read_csv(CSV_URL)

print("Dataset shape:", df.shape)
display(df.head())
df.info()

Dataset shape: (10521, 12)


,Entity,Code,Year,Number_yaws,polio_cases,cases_guinea_worm,Number_rabies,Number_malaria,Number_hiv,Number_tuberculosis,Number_smallpox,Number_cholera_cases
0,Afghanistan,AFG,1980,NaN,6160.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,AFG,1981,NaN,5859.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,1982,NaN,9730.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,AFG,1983,NaN,13937.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,AFG,1984,NaN,3864.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10521 entries, 0 to 10520
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Entity                10521 non-null  object 
 1   Code                  9393 non-null   object 
 2   Year                  10521 non-null  int64  
 3   Number_yaws           74 non-null     float64
 4   polio_cases           7610 non-null   float64
 5   cases_guinea_worm     8428 non-null   float64
 6   Number_rabies         6780 non-null   float64
 7   Number_malaria        6780 non-null   float64
 8   Number_hiv            6780 non-null   float64
 9   Number_tuberculosis   6780 non-null   float64
 10  Number_smallpox       104 non-null    float64
 11  Number_cholera_cases  2967 non-null   float64
dtypes: float64(9), int64(1), object(2)
memory usage: 986.5+ KB


In [59]:
df.to_sql(
    name="infectious_cases",
    con=engine,
    schema="pandemic",
    if_exists="replace",
    index=False,
    chunksize=5000
)

print("CSV imported successfully.")

CSV imported successfully.


In [60]:
display(pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT "Entity") AS unique_entities,
        MIN("Year") AS min_year,
        MAX("Year") AS max_year,
        COUNT(*) FILTER (
            WHERE "Number_rabies" IS NULL
        ) AS missing_rabies
    FROM pandemic.infectious_cases;
""", engine))

,total_rows,unique_entities,min_year,max_year,missing_rabies
0,10521,245,1920,2023,3741


## Task 2. Database Normalization to Third Normal Form (3NF)

Separate unique entity-code combinations from yearly infectious disease statistics.

In [61]:
with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS pandemic.infectious_cases_normalized;
        DROP TABLE IF EXISTS pandemic.entities;

        CREATE TABLE pandemic.entities (
            id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
            entity_name TEXT NOT NULL,
            entity_code TEXT,
            UNIQUE NULLS NOT DISTINCT (entity_name, entity_code)
        );

        INSERT INTO pandemic.entities (entity_name, entity_code)
        SELECT DISTINCT
            "Entity",
            "Code"
        FROM pandemic.infectious_cases
        ORDER BY "Entity", "Code";
    """))

print("Entities table created successfully.")

Entities table created successfully.


In [62]:
#Creating normalized fact table
with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE pandemic.infectious_cases_normalized (
            entity_id INTEGER NOT NULL
                REFERENCES pandemic.entities(id),

            year INTEGER NOT NULL,

            number_yaws DOUBLE PRECISION,
            polio_cases DOUBLE PRECISION,
            cases_guinea_worm DOUBLE PRECISION,
            number_rabies DOUBLE PRECISION,
            number_malaria DOUBLE PRECISION,
            number_hiv DOUBLE PRECISION,
            number_tuberculosis DOUBLE PRECISION,
            number_smallpox DOUBLE PRECISION,
            number_cholera_cases DOUBLE PRECISION,

            PRIMARY KEY (entity_id, year)
        );

        INSERT INTO pandemic.infectious_cases_normalized (
            entity_id,
            year,
            number_yaws,
            polio_cases,
            cases_guinea_worm,
            number_rabies,
            number_malaria,
            number_hiv,
            number_tuberculosis,
            number_smallpox,
            number_cholera_cases
        )
        SELECT
            e.id,
            r."Year",
            r."Number_yaws",
            r.polio_cases,
            r.cases_guinea_worm,
            r."Number_rabies",
            r."Number_malaria",
            r."Number_hiv",
            r."Number_tuberculosis",
            r."Number_smallpox",
            r."Number_cholera_cases"
        FROM pandemic.infectious_cases AS r
        JOIN pandemic.entities AS e
            ON e.entity_name = r."Entity"
           AND e.entity_code IS NOT DISTINCT FROM r."Code";
    """))

print("Normalized disease table created successfully.")

Normalized disease table created successfully.


In [63]:
display(pd.read_sql("""
    SELECT
        (SELECT COUNT(*)
         FROM pandemic.infectious_cases) AS original_rows,

        (SELECT COUNT(*)
         FROM pandemic.entities) AS unique_entities,

        (SELECT COUNT(*)
         FROM pandemic.infectious_cases_normalized) AS normalized_rows,

        (SELECT COUNT(*)
         FROM pandemic.infectious_cases_normalized f
         LEFT JOIN pandemic.entities e
             ON f.entity_id = e.id
         WHERE e.id IS NULL) AS orphan_records;
""", engine))

,original_rows,unique_entities,normalized_rows,orphan_records
0,10521,245,10521,0


In [64]:
#Check if we can reproduce table
display(pd.read_sql("""
    SELECT
        e.entity_name,
        e.entity_code,
        f.year,
        f.polio_cases,
        f.number_rabies
    FROM pandemic.infectious_cases_normalized AS f
    JOIN pandemic.entities AS e
        ON f.entity_id = e.id
    ORDER BY e.entity_name, f.year
    LIMIT 10;
""", engine))

,entity_name,entity_code,year,polio_cases,number_rabies
0,Afghanistan,AFG,1960,NaN,None
1,Afghanistan,AFG,1965,NaN,None
2,Afghanistan,AFG,1980,6160.0,None
3,Afghanistan,AFG,1981,5859.0,None
4,Afghanistan,AFG,1982,9730.0,None
5,Afghanistan,AFG,1983,13937.0,None
6,Afghanistan,AFG,1984,3864.0,None
7,Afghanistan,AFG,1985,13867.0,None
8,Afghanistan,AFG,1986,12901.0,None
9,Afghanistan,AFG,1987,4396.0,None


### Task 2 — Database Normalization Summary

The original `infectious_cases` table was normalized into two related tables: `entities` and `infectious_cases_normalized`.

The `entities` table contains 245 unique country/region and code combinations, while the normalized fact table stores yearly disease statistics using a foreign key (`entity_id`).

Validation confirmed that all 10,521 original records were preserved, with zero orphan records. The normalized schema reduces redundancy and maintains referential integrity.

## Task 3. Rabies Statistics Analysis

Calculate average, minimum, maximum, and total rabies cases by entity, ordered by average cases in descending order.

In [65]:
display(pd.read_sql("""
    SELECT
        e.entity_name,
        e.entity_code,
        ROUND(AVG(f.number_rabies)::NUMERIC, 2) AS avg_rabies,
        MIN(f.number_rabies) AS min_rabies,
        MAX(f.number_rabies) AS max_rabies,
        SUM(f.number_rabies) AS total_rabies
    FROM pandemic.infectious_cases_normalized AS f
    JOIN pandemic.entities AS e
        ON f.entity_id = e.id
    WHERE f.number_rabies IS NOT NULL
    GROUP BY
        e.id,
        e.entity_name,
        e.entity_code
    ORDER BY avg_rabies DESC
    LIMIT 10;
""", engine))

,entity_name,entity_code,avg_rabies,min_rabies,max_rabies,total_rabies
0,World,OWID_WRL,20192.37,14075.5080,24744.6580,605771.1110
1,Lower Middle Income (WB),None,15193.96,10202.5280,19182.7950,455818.7815
2,South Asia (WB),None,11729.89,7271.2754,15361.8780,351896.6626
3,South-East Asia Region (WHO),None,11424.33,6806.0127,15641.9580,342729.8068
4,G20,None,10189.05,6339.0796,13164.8810,305671.3801
5,India,IND,8599.17,5425.8726,11121.1420,257975.1984
6,Sub-Saharan Africa (WB),None,4748.27,4356.8184,5017.9770,142448.0427
7,African Region (WHO),None,4649.70,4283.0730,4917.7840,139491.0713
8,Low Income (WB),None,3568.11,3014.6170,3873.5330,107043.1635
9,East Asia & Pacific (WB),None,3427.46,2009.9395,4591.0107,102823.8340


## Task 4. Year Difference Calculation

In [66]:
display(pd.read_sql("""
    SELECT
        e.entity_name,
        e.entity_code,
        f.year,

        MAKE_DATE(f.year, 1, 1) AS year_start_date,

        CURRENT_DATE AS current_date,

        EXTRACT(
            YEAR FROM AGE(
                CURRENT_DATE,
                MAKE_DATE(f.year, 1, 1)
            )
        )::INTEGER AS years_difference

    FROM pandemic.infectious_cases_normalized AS f
    JOIN pandemic.entities AS e
        ON f.entity_id = e.id

    ORDER BY e.entity_name, f.year
    LIMIT 10;
""", engine))

,entity_name,entity_code,year,year_start_date,current_date,years_difference
0,Afghanistan,AFG,1960,1960-01-01,2026-10-18,66
1,Afghanistan,AFG,1965,1965-01-01,2026-10-18,61
2,Afghanistan,AFG,1980,1980-01-01,2026-10-18,46
3,Afghanistan,AFG,1981,1981-01-01,2026-10-18,45
4,Afghanistan,AFG,1982,1982-01-01,2026-10-18,44
5,Afghanistan,AFG,1983,1983-01-01,2026-10-18,43
6,Afghanistan,AFG,1984,1984-01-01,2026-10-18,42
7,Afghanistan,AFG,1985,1985-01-01,2026-10-18,41
8,Afghanistan,AFG,1986,1986-01-01,2026-10-18,40
9,Afghanistan,AFG,1987,1987-01-01,2026-10-18,39


### Task 4 — Date Calculations

The `MAKE_DATE()` function was used to construct January 1 for each observation year. `CURRENT_DATE` provides the reference date, while `AGE()` and `EXTRACT(YEAR FROM ...)` calculate the difference in complete calendar years.

This approach demonstrates PostgreSQL date functions without modifying the original dataset.

## Task 5. User-Defined SQL Functions

### 5.1 Calculate Years Difference

In [67]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE OR REPLACE FUNCTION pandemic.calculate_years_difference(
            input_year INTEGER
        )
        RETURNS INTEGER
        LANGUAGE SQL
        STABLE
        AS $$
            SELECT EXTRACT(
                YEAR FROM AGE(
                    CURRENT_DATE,
                    MAKE_DATE(input_year, 1, 1)
                )
            )::INTEGER;
        $$;
    """))

print("Years difference function created successfully.")

Years difference function created successfully.


In [68]:
display(pd.read_sql("""
    SELECT
        e.entity_name,
        f.year,
        MAKE_DATE(f.year, 1, 1) AS year_start_date,
        CURRENT_DATE AS current_date,
        pandemic.calculate_years_difference(f.year)
            AS years_difference
    FROM pandemic.infectious_cases_normalized AS f
    JOIN pandemic.entities AS e
        ON f.entity_id = e.id
    ORDER BY e.entity_name, f.year
    LIMIT 10;
""", engine))

,entity_name,year,year_start_date,current_date,years_difference
0,Afghanistan,1960,1960-01-01,2026-10-18,66
1,Afghanistan,1965,1965-01-01,2026-10-18,61
2,Afghanistan,1980,1980-01-01,2026-10-18,46
3,Afghanistan,1981,1981-01-01,2026-10-18,45
4,Afghanistan,1982,1982-01-01,2026-10-18,44
5,Afghanistan,1983,1983-01-01,2026-10-18,43
6,Afghanistan,1984,1984-01-01,2026-10-18,42
7,Afghanistan,1985,1985-01-01,2026-10-18,41
8,Afghanistan,1986,1986-01-01,2026-10-18,40
9,Afghanistan,1987,1987-01-01,2026-10-18,39


### 5.2 Calculate Average Cases per Period

In [69]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE OR REPLACE FUNCTION pandemic.calculate_period_average(
            annual_cases DOUBLE PRECISION,
            divisor INTEGER
        )
        RETURNS DOUBLE PRECISION
        LANGUAGE SQL
        IMMUTABLE
        AS $$
            SELECT annual_cases / NULLIF(divisor, 0);
        $$;
    """))

print("Period average function created successfully.")

Period average function created successfully.


In [70]:
display(pd.read_sql("""
    SELECT
        e.entity_name,
        f.year,
        f.number_rabies AS annual_rabies,

        ROUND(
            pandemic.calculate_period_average(
                f.number_rabies, 12
            )::NUMERIC, 2
        ) AS monthly_avg,

        ROUND(
            pandemic.calculate_period_average(
                f.number_rabies, 4
            )::NUMERIC, 2
        ) AS quarterly_avg,

        ROUND(
            pandemic.calculate_period_average(
                f.number_rabies, 2
            )::NUMERIC, 2
        ) AS half_year_avg

    FROM pandemic.infectious_cases_normalized AS f
    JOIN pandemic.entities AS e
        ON f.entity_id = e.id
    WHERE f.number_rabies IS NOT NULL
    ORDER BY e.entity_name, f.year
    LIMIT 10;
""", engine))

,entity_name,year,annual_rabies,monthly_avg,quarterly_avg,half_year_avg
0,Afghanistan,1990,2.145772,0.18,0.54,1.07
1,Afghanistan,1991,2.320975,0.19,0.58,1.16
2,Afghanistan,1992,2.618041,0.22,0.65,1.31
3,Afghanistan,1993,2.853117,0.24,0.71,1.43
4,Afghanistan,1994,2.989987,0.25,0.75,1.49
5,Afghanistan,1995,3.126361,0.26,0.78,1.56
6,Afghanistan,1996,3.281437,0.27,0.82,1.64
7,Afghanistan,1997,3.446122,0.29,0.86,1.72
8,Afghanistan,1998,3.557025,0.30,0.89,1.78
9,Afghanistan,1999,3.607037,0.30,0.90,1.80


In [71]:
display(pd.read_sql("""
    SELECT
        pandemic.calculate_years_difference(1996)
            AS years_since_1996,

        pandemic.calculate_period_average(120, 12)
            AS monthly_avg,

        pandemic.calculate_period_average(120, 4)
            AS quarterly_avg,

        pandemic.calculate_period_average(120, 2)
            AS half_year_avg,

        pandemic.calculate_period_average(120, 0)
            AS zero_division_check;
""", engine))

,years_since_1996,monthly_avg,quarterly_avg,half_year_avg,zero_division_check
0,30,10.0,30.0,60.0,None


### Task 5 — User-Defined SQL Functions Summary

Two PostgreSQL SQL functions were implemented and validated.

The `calculate_years_difference()` function calculates complete calendar years between January 1 of a given year and the current date, producing results consistent with Task 4.

The `calculate_period_average()` function converts annual disease values into monthly, quarterly, and semiannual averages using divisors of 12, 4, and 2. It also safely handles division by zero by returning `NULL`.

Both functions were tested with sample inputs and applied to the normalized infectious disease dataset.